# xlsm Extractor
Extracts **E.1. Table 5** and the **H.1 Manifest** sheet from an `.xlsm` / `.xlsx` workbook.

Configure the options in the **Config** cell, then run all cells.

In [ ]:
# Install dependencies if needed
# !pip install openpyxl pandas tabulate openpyxl

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path("extractor.py").resolve().parent))
from extractor import extract, _render, _save, _save_excel

In [ ]:
# ── CONFIG ────────────────────────────────────────────────────────────────────

# Path to your workbook
FILE = "report.xlsm"

# Sheet names (leave as None to auto-detect)
E1_SHEET      = None          # e.g. "E1 Norm+Pool"
MANIFEST_SHEET = None         # e.g. "H.1 Manifest"

# Table label to search for in the E1 sheet
TABLE_LABEL   = "E.1. Table 5"

# Treat first data row as data rather than column headers?
NO_HEADER     = False

# Which table(s) to show/export: "e1" | "manifest" | "both"
SHOW          = "both"

# Display format for preview: "table" | "csv" | "tsv" | "json" | "markdown"
DISPLAY_FORMAT = "table"

# Limit preview rows (None = all)
MAX_ROWS      = 20

# ── EXPORT ────────────────────────────────────────────────────────────────────
# Set OUT_DIR or OUT_FILE to export; leave both as None to only preview.

# Export format: "table" | "csv" | "tsv" | "json" | "markdown" | "excel"
# Use "excel" to write both tables into a single .xlsx workbook.
EXPORT_FORMAT = "excel"

# Save all output files to this directory (ignored if OUT_FILE is set)
OUT_DIR       = None          # e.g. "./output"

# Save to a specific file (only used when SHOW != "both" OR EXPORT_FORMAT == "excel")
OUT_FILE      = None          # e.g. "results.xlsx"

# ─────────────────────────────────────────────────────────────────────────────

In [ ]:
# Extract
e1_df, manifest_df = extract(
    FILE,
    e1_sheet=E1_SHEET,
    manifest_sheet=MANIFEST_SHEET,
    table_label=TABLE_LABEL,
    no_header=NO_HEADER,
)
print(f"E.1. Table 5  → {e1_df.shape[0]} rows × {e1_df.shape[1]} cols")
print(f"H.1 Manifest  → {manifest_df.shape[0]} rows × {manifest_df.shape[1]} cols")

In [ ]:
# Preview
import pandas as pd
pd.set_option("display.max_columns", None)

show_e1       = SHOW in ("e1", "both")
show_manifest = SHOW in ("manifest", "both")

if show_e1:
    print("=== E.1. Table 5 ===")
    if DISPLAY_FORMAT == "table":
        display(e1_df.head(MAX_ROWS) if MAX_ROWS else e1_df)
    else:
        print(_render(e1_df, DISPLAY_FORMAT, MAX_ROWS))

if show_manifest:
    print("=== H.1 Manifest ===")
    if DISPLAY_FORMAT == "table":
        display(manifest_df.head(MAX_ROWS) if MAX_ROWS else manifest_df)
    else:
        print(_render(manifest_df, DISPLAY_FORMAT, MAX_ROWS))

In [ ]:
# Export (skip this cell if OUT_DIR and OUT_FILE are both None)
import pathlib

if OUT_DIR or OUT_FILE:
    ext_map = {"table": "txt", "csv": "csv", "tsv": "tsv",
               "json": "json", "markdown": "md", "excel": "xlsx"}
    ext = ext_map[EXPORT_FORMAT]

    all_tables = [
        ("E.1. Table 5", "e1_table5", e1_df),
        ("H.1 Manifest", "manifest",  manifest_df),
    ]
    export_tables = [
        t for t in all_tables
        if (t[1] == "e1_table5" and show_e1) or
           (t[1] == "manifest"  and show_manifest)
    ]

    if EXPORT_FORMAT == "excel":
        dest = pathlib.Path(OUT_FILE) if OUT_FILE else \
               pathlib.Path(OUT_DIR) / f"output.{ext}"
        print("Exporting [Excel workbook] (both tables) ...")
        _save_excel(all_tables, dest)   # always writes both sheets
    else:
        for label, stem, df in export_tables:
            dest = pathlib.Path(OUT_FILE) if OUT_FILE else \
                   pathlib.Path(OUT_DIR) / f"{stem}.{ext}"
            print(f"Exporting [{label}] ...")
            _save(df, dest, EXPORT_FORMAT)
else:
    print("No export — set OUT_DIR or OUT_FILE in the Config cell to save files.")